# Public data experiments
This is the notebook to run the public data experiment

In [ ]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name
from src.model.functions import save_model
try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd

In [ ]:
# Define the data directory

data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")

# Ask user for dataset name (without .csv extension)
dataset_name = "german"
dataset_path = data_dir / "input" / f"{dataset_name}.csv"

# Check if file exists and read it using the helper from src.csv_config
if dataset_path.exists():
    # Use the dataset-specific defaults (german uses tab separator)
    df = read_csv_by_name(str(dataset_path), dataset_name)
    df.head()
else:
    print(f"File not found: {dataset_path}")

In [ ]:
# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM"]
product = "default"          # hyper‑search space selector

# prepare column lists once
feature_cols     = df.columns.tolist()
numerical_cols   = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()
ids_cols         = []
target_cols      = ["target"]
exclude_cols     = []

results = []                  # collect stats if you like

for model_choice in model_choices:
    print(f"\n=== training {model_choice} ===")
    pipeline, best_params, best_score = get_train_model(
        data=df,
        feature_cols=feature_cols,
        numerical_cols=numerical_cols,
        categorical_cols=categorical_cols,
        ids_cols=ids_cols,
        target_cols=target_cols,
        exclude_cols=exclude_cols,
        product=product,
        algorithm=model_choice,
        test_mode=True,          # set False for full search
    )

    print("Best hyperparameters:", best_params)
    print("Best CV score:", best_score)

    # save the trained model object
    model_path = data_dir / "output" / f"{dataset_name}_{model_choice}.pkl"
    save_model(str(model_path), pipeline, environment="local")
    print(f"trained pipeline saved to {model_path}")

    results.append(
        {"model": model_choice, "params": best_params, "score": best_score}
    )

# optional: look at all results afterwards
import pandas as pd
pd.DataFrame(results)